In [1]:
import sys
import torch

print("Python usado:", sys.executable)
print("Torch version:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())
print("CUDA version en PyTorch:", torch.version.cuda)
print("Torch cargado desde:", torch.__file__)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Python usado: c:\Users\eliza\Semestre8\AplicacionesAvanzadas\Oscar\Reto\PlagIArismDetector\.venv\Scripts\python.exe
Torch version: 2.13.0.dev20260527+cu132
CUDA disponible: True
CUDA version en PyTorch: 13.2
Torch cargado desde: c:\Users\eliza\Semestre8\AplicacionesAvanzadas\Oscar\Reto\PlagIArismDetector\.venv\Lib\site-packages\torch\__init__.py
GPU: NVIDIA GeForce RTX 5060 Laptop GPU


In [2]:
import os
import itertools
import pandas as pd
import torch
import torch.nn.functional as F

from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity

c:\Users\eliza\Semestre8\AplicacionesAvanzadas\Oscar\Reto\PlagIArismDetector\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
base_path = r"C:\Users\eliza\Semestre8\AplicacionesAvanzadas\Oscar\Reto\AI-SOCO\SOCO"

train_csv_path = os.path.join(base_path, "train.csv")
train_folder = os.path.join(base_path, "train")

model_name = "neulab/codebert-cpp"

batch_size = 8
max_length = 512

output_embeddings_path = os.path.join("C:\\Users\\eliza\\Semestre8\\AplicacionesAvanzadas\\Oscar\\Reto", "codebert_cpp_embeddings.pt")
output_similarity_path = os.path.join("C:\\Users\\eliza\\Semestre8\\AplicacionesAvanzadas\\Oscar\\Reto", "similaridades_codebert_cpp.csv")

In [4]:
train_df = pd.read_csv(train_csv_path)

print("Columnas del CSV:")
print(train_df.columns)

# Validación mínima
required_columns = {"pid", "uid"}

if not required_columns.issubset(set(train_df.columns)):
    raise ValueError(
        f"El CSV debe contener las columnas {required_columns}. "
        f"Columnas encontradas: {set(train_df.columns)}"
    )

Columnas del CSV:
Index(['uid', 'pid'], dtype='str')


In [ ]:
codigos = []
labels = []
file_ids = []

for _, row in train_df.iterrows():

    nombre_archivo = str(row["pid"])
    ruta_archivo = os.path.join(train_folder, nombre_archivo)

    try:
        with open(ruta_archivo, "r", encoding="utf-8", errors="ignore") as f:
            codigo = f.read()

        codigo = codigo.replace("\t", " ")
        codigo = codigo.strip()

        if len(codigo) == 0:
            print(f"Archivo vacío ignorado: {ruta_archivo}")
            continue

        codigos.append(codigo)
        labels.append(row["uid"])
        file_ids.append(nombre_archivo)

    except Exception as e:
        print(f"Error leyendo {ruta_archivo}: {e}")


print(f"\nTotal de códigos cargados: {len(codigos)}")


Total de códigos cargados: 50000


In [6]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

if torch.cuda.is_available():
    device = torch.device("cuda")
    print("\nUsando GPU:")
    print(torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("\nGPU no disponible, usando CPU")

model.to(device)
model.eval()



Loading weights: 100%|██████████| 197/197 [00:00<00:00, 16310.91it/s]
[transformers] RobertaModel LOAD REPORT from: neulab/codebert-cpp
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Usando GPU:
NVIDIA GeForce RTX 5060 Laptop GPU


RobertaModel(
  (embeddings): RobertaEmbeddings(
    (word_embeddings): Embedding(50265, 768, padding_idx=1)
    (token_type_embeddings): Embedding(1, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
    (position_embeddings): Embedding(514, 768, padding_idx=1)
  )
  (encoder): RobertaEncoder(
    (layer): ModuleList(
      (0-11): 12 x RobertaLayer(
        (attention): RobertaAttention(
          (self): RobertaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): RobertaSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=Tru

In [ ]:
def mean_pooling(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()

    summed = torch.sum(last_hidden_state * mask, dim=1)

    counts = torch.clamp(mask.sum(dim=1), min=1e-9)

    return summed / counts


In [ ]:
all_embeddings = []

total_batches = (len(codigos) + batch_size - 1) // batch_size

with torch.no_grad():

    for i in range(0, len(codigos), batch_size):

        batch_num = (i // batch_size) + 1
        batch_codigos = codigos[i:i + batch_size]

        tokens = tokenizer(
            batch_codigos,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )

        input_ids = tokens["input_ids"].to(device)
        attention_mask = tokens["attention_mask"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        batch_embeddings = mean_pooling(
            outputs.last_hidden_state,
            attention_mask
        )

        batch_embeddings = F.normalize(batch_embeddings, p=2, dim=1)

        all_embeddings.append(batch_embeddings.cpu())

        print(f"Batch {batch_num}/{total_batches} procesado")

        if device.type == "cuda":
            torch.cuda.empty_cache()


embeddings = torch.cat(all_embeddings, dim=0)

print("\nShape embeddings:")
print(embeddings.shape)

torch.save(
    {
        "embeddings": embeddings,
        "file_ids": file_ids,
        "labels": labels,
        "model_name": model_name
    },
    output_embeddings_path
)

print(f"\nEmbeddings guardados en:")
print(output_embeddings_path)


Batch 1/6250 procesado
Batch 2/6250 procesado
Batch 3/6250 procesado
Batch 4/6250 procesado
Batch 5/6250 procesado
Batch 6/6250 procesado
Batch 7/6250 procesado
Batch 8/6250 procesado
Batch 9/6250 procesado
Batch 10/6250 procesado
Batch 11/6250 procesado
Batch 12/6250 procesado
Batch 13/6250 procesado
Batch 14/6250 procesado
Batch 15/6250 procesado
Batch 16/6250 procesado
Batch 17/6250 procesado
Batch 18/6250 procesado
Batch 19/6250 procesado
Batch 20/6250 procesado
Batch 21/6250 procesado
Batch 22/6250 procesado
Batch 23/6250 procesado
Batch 24/6250 procesado
Batch 25/6250 procesado
Batch 26/6250 procesado
Batch 27/6250 procesado
Batch 28/6250 procesado
Batch 29/6250 procesado
Batch 30/6250 procesado
Batch 31/6250 procesado
Batch 32/6250 procesado
Batch 33/6250 procesado
Batch 34/6250 procesado
Batch 35/6250 procesado
Batch 36/6250 procesado
Batch 37/6250 procesado
Batch 38/6250 procesado
Batch 39/6250 procesado
Batch 40/6250 procesado
Batch 41/6250 procesado
Batch 42/6250 procesado
B

In [ ]:
import csv
import torch

output_similarity_path = os.path.join(
    "C:\\Users\\eliza\\Semestre8\\AplicacionesAvanzadas\\Oscar\\Reto",
    "similaridades_sospechosas_codebert_cpp.csv"
)

threshold = 0.80

sim_batch_size = 256

embeddings_cpu = embeddings.float()

num_codigos = embeddings_cpu.shape[0]

print(f"\nCalculando similitudes por bloques...")
print(f"Total de códigos: {num_codigos}")
print(f"Umbral de similitud: {threshold}")

total_guardados = 0

with open(output_similarity_path, "w", newline="", encoding="utf-8") as csvfile:

    writer = csv.writer(csvfile)

    writer.writerow([
        "file_1",
        "file_2",
        "uid_1",
        "uid_2",
        "same_uid",
        "similarity"
    ])

    for start in range(0, num_codigos, sim_batch_size):

        end = min(start + sim_batch_size, num_codigos)

        batch_emb = embeddings_cpu[start:end]

        sim_block = torch.matmul(batch_emb, embeddings_cpu.T)

        for local_i in range(end - start):

            global_i = start + local_i

            sim_block[local_i, :global_i + 1] = -1.0

            indices = torch.where(sim_block[local_i] >= threshold)[0]

            for j in indices.tolist():

                similarity = sim_block[local_i, j].item()

                same_uid = labels[global_i] == labels[j]

                writer.writerow([
                    file_ids[global_i],
                    file_ids[j],
                    labels[global_i],
                    labels[j],
                    same_uid,
                    similarity
                ])

                total_guardados += 1

        print(f"Bloque {start} - {end} procesado")

print("\nProceso terminado.")
print(f"Pares sospechosos guardados: {total_guardados}")
print(f"Archivo generado: {output_similarity_path}")


Calculando similitudes por bloques...
Total de códigos: 50000
Umbral de similitud: 0.8
Bloque 0 - 256 procesado
Bloque 256 - 512 procesado
Bloque 512 - 768 procesado
Bloque 768 - 1024 procesado
Bloque 1024 - 1280 procesado
Bloque 1280 - 1536 procesado
Bloque 1536 - 1792 procesado
Bloque 1792 - 2048 procesado
Bloque 2048 - 2304 procesado
Bloque 2304 - 2560 procesado
Bloque 2560 - 2816 procesado
Bloque 2816 - 3072 procesado
Bloque 3072 - 3328 procesado
Bloque 3328 - 3584 procesado
Bloque 3584 - 3840 procesado
Bloque 3840 - 4096 procesado
Bloque 4096 - 4352 procesado
Bloque 4352 - 4608 procesado
Bloque 4608 - 4864 procesado
Bloque 4864 - 5120 procesado
Bloque 5120 - 5376 procesado
Bloque 5376 - 5632 procesado
Bloque 5632 - 5888 procesado
Bloque 5888 - 6144 procesado
Bloque 6144 - 6400 procesado
Bloque 6400 - 6656 procesado
Bloque 6656 - 6912 procesado
Bloque 6912 - 7168 procesado
Bloque 7168 - 7424 procesado
Bloque 7424 - 7680 procesado
Bloque 7680 - 7936 procesado
Bloque 7936 - 8192 pro